In [1]:
import os
import time
import pickle
import numpy as np
import pandas as pd

from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# ============================================================
# CONFIG
# ============================================================

SEED = 42
RNG = np.random.default_rng(SEED)

DATASETS_ROOT = "/var/new_homes/julio/mestrado/mestrado-dyssyn/datasets/binary"
OUTPUT_DIR = "moss_outputs"

# Geração MoSS
N_SAMPLES = 100
N_PREVALENCES = 19                  # alinhado ao APP (0.05..0.95)
N_MERGES = 13                       # evita merge extremo
N_CURVES = 30

# Para calibrar a distribuição real
MAX_DATASETS_FOR_CALIB = 80         # ajuste conforme tempo
TEST_SIZE = 0.5

# ============================================================
# UTILS
# ============================================================

def to_beta_params_from_mean_var(mu, var, eps=1e-6):
    """
    Converte média/variância para alpha,beta da Beta(mu,var).
    Garante estabilidade numérica.
    """
    mu = float(np.clip(mu, eps, 1 - eps))
    max_var = mu * (1 - mu) - eps
    var = float(np.clip(var, eps, max_var))

    k = mu * (1 - mu) / var - 1.0
    # k precisa ser > 0
    k = max(k, 1e-3)

    alpha = mu * k
    beta = (1 - mu) * k
    return alpha, beta


def safe_var(x):
    v = float(np.var(x))
    return max(v, 1e-6)


def load_binary_dataset(csv_path):
    df = pd.read_csv(csv_path)
    y = df.iloc[:, -1].values
    X = df.iloc[:, :-1].values

    classes = np.unique(y)
    if len(classes) != 2:
        return None, None

    # classe positiva = max label
    y = (y == classes.max()).astype(int)
    return X, y


# ============================================================
# ETAPA 1: CALIBRAR DISTRIBUIÇÃO REAL DE SCORES
# ============================================================

@dataclass
class ScoreStats:
    mu_neg: float
    var_neg: float
    mu_pos: float
    var_pos: float
    sep: float  # separabilidade = mu_pos - mu_neg


def collect_real_score_stats(
    datasets_root=DATASETS_ROOT,
    max_datasets=MAX_DATASETS_FOR_CALIB,
    seed=SEED
):
    files = sorted(f for f in os.listdir(datasets_root) if f.endswith(".csv"))
    if len(files) == 0:
        raise RuntimeError(f"Nenhum CSV em {datasets_root}")

    stats = []
    used = 0

    for f in files:
        if used >= max_datasets:
            break

        path = os.path.join(datasets_root, f)
        X, y = load_binary_dataset(path)
        if X is None:
            continue

        # skip datasets degenerados
        if len(np.unique(y)) < 2:
            continue

        try:
            Xtr, Xte, ytr, yte = train_test_split(
                X, y,
                test_size=TEST_SIZE,
                stratify=y,
                random_state=seed
            )
        except Exception:
            continue

        scaler = StandardScaler().fit(Xtr)
        Xtr = scaler.transform(Xtr)
        Xte = scaler.transform(Xte)

        clf = RandomForestClassifier(
            n_estimators=300,
            random_state=seed,
            n_jobs=-1
        )
        clf.fit(Xtr, ytr)

        scores = clf.predict_proba(Xte)[:, 1]
        s_neg = scores[yte == 0]
        s_pos = scores[yte == 1]

        if len(s_neg) < 10 or len(s_pos) < 10:
            continue

        mu_neg = float(np.mean(s_neg))
        mu_pos = float(np.mean(s_pos))
        var_neg = safe_var(s_neg)
        var_pos = safe_var(s_pos)

        # só aceita perfil minimamente consistente
        if mu_pos <= mu_neg:
            continue

        stats.append(ScoreStats(
            mu_neg=mu_neg,
            var_neg=var_neg,
            mu_pos=mu_pos,
            var_pos=var_pos,
            sep=mu_pos - mu_neg
        ))
        used += 1

    if len(stats) == 0:
        raise RuntimeError("Não foi possível calibrar stats reais dos datasets binários.")

    return stats


def robust_pool_stats(score_stats):
    """
    Agregação robusta (medianas) para reduzir efeito de outliers.
    """
    mu_neg = float(np.median([s.mu_neg for s in score_stats]))
    var_neg = float(np.median([s.var_neg for s in score_stats]))
    mu_pos = float(np.median([s.mu_pos for s in score_stats]))
    var_pos = float(np.median([s.var_pos for s in score_stats]))
    sep = float(np.median([s.sep for s in score_stats]))

    return ScoreStats(mu_neg, var_neg, mu_pos, var_pos, sep)


# ============================================================
# ETAPA 2: GERADOR MOSS MAIS REPRESENTATIVO
# ============================================================

def moss_binary_beta_calibrated(
    n_samples: int,
    p_pos: float,
    merge: float,
    pooled: ScoreStats
):
    """
    Gera scores binários [p_neg, p_pos] calibrados em stats reais.
    merge em [0,1): quanto maior, menor separação e maior sobreposição.
    """
    merge = float(np.clip(merge, 0.0, 0.95))  # evita colapso extremo

    # Interpola médias em direção a 0.5 conforme merge
    mu_neg = (1 - merge) * pooled.mu_neg + merge * 0.5
    mu_pos = (1 - merge) * pooled.mu_pos + merge * 0.5

    # Aumenta variância com merge (mais incerteza), mas controlado
    # fator de inflar variância: 1x -> ~3x
    var_factor = 1.0 + 2.0 * merge
    var_neg = pooled.var_neg * var_factor
    var_pos = pooled.var_pos * var_factor

    # Converte para params Beta
    a_neg, b_neg = to_beta_params_from_mean_var(mu_neg, var_neg)
    a_pos, b_pos = to_beta_params_from_mean_var(mu_pos, var_pos)

    n_pos = int(np.floor(n_samples * p_pos))
    n_neg = n_samples - n_pos

    scores = np.zeros((n_samples, 2), dtype=float)

    if n_neg > 0:
        s_neg = RNG.beta(a_neg, b_neg, size=n_neg)
        scores[:n_neg, 1] = s_neg
        scores[:n_neg, 0] = 1.0 - s_neg

    if n_pos > 0:
        s_pos = RNG.beta(a_pos, b_pos, size=n_pos)
        scores[n_neg:, 1] = s_pos
        scores[n_neg:, 0] = 1.0 - s_pos

    RNG.shuffle(scores)
    return scores


def gerar_distribuicoes_moss_binario_representativo(
    n_samples=N_SAMPLES,
    n_prevalences=N_PREVALENCES,
    n_merges=N_MERGES,
    n_curves=N_CURVES,
    save_path="moss_outputs/moss_binario_lite.pkl"
):
    # grade de prevalências alinhada ao APP
    prevalences = np.linspace(0.05, 0.95, n_prevalences)
    # evita 1.0 para não centralizar tudo em 0.5
    merges = np.linspace(0.0, 0.95, n_merges)

    print("🔎 Coletando estatísticas reais de scores...")
    stats = collect_real_score_stats()
    pooled = robust_pool_stats(stats)
    print(f"   Datasets usados na calibração: {len(stats)}")
    print(f"   mu_neg={pooled.mu_neg:.4f}, mu_pos={pooled.mu_pos:.4f}, sep={pooled.sep:.4f}")

    synthetic_distributions = {}
    total = len(prevalences) * len(merges)
    count = 0

    print(f"\n🚀 Gerando MoSS BINÁRIO calibrado -> {save_path}")
    t0 = time.perf_counter()

    for p in prevalences:
        alpha_key = (round(1 - p, 4), round(p, 4))
        for m in merges:
            curves = [
                moss_binary_beta_calibrated(
                    n_samples=n_samples,
                    p_pos=p,
                    merge=float(m),
                    pooled=pooled
                )
                for _ in range(n_curves)
            ]
            # mantém o mesmo padrão de chave
            synthetic_distributions[(alpha_key, round(float(m), 4))] = curves

            count += 1
            if count % 50 == 0 or count == total:
                print(f"   Progresso: [{count}/{total}] blocos")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "wb") as f:
        pickle.dump(synthetic_distributions, f, protocol=pickle.HIGHEST_PROTOCOL)

    dt = time.perf_counter() - t0
    print(f"\n✅ Finalizado em {dt/60:.2f} min")
    print(f"🏁 Arquivo gerado: {save_path}")


if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    file_path = os.path.join(OUTPUT_DIR, "moss_binario_lite.pkl")

    gerar_distribuicoes_moss_binario_representativo(
        n_samples=N_SAMPLES,
        n_prevalences=N_PREVALENCES,
        n_merges=N_MERGES,
        n_curves=N_CURVES,
        save_path=file_path
    )


🔎 Coletando estatísticas reais de scores...
   Datasets usados na calibração: 80
   mu_neg=0.1729, mu_pos=0.8313, sep=0.5648

🚀 Gerando MoSS BINÁRIO calibrado -> moss_outputs/moss_binario_lite.pkl
   Progresso: [50/247] blocos
   Progresso: [100/247] blocos
   Progresso: [150/247] blocos
   Progresso: [200/247] blocos
   Progresso: [247/247] blocos

✅ Finalizado em 0.02 min
🏁 Arquivo gerado: moss_outputs/moss_binario_lite.pkl
